# Atividade Prática — Aula 3: Limpeza, Preparação e Qualidade dos Dados com Pandas

Esta atividade foi construída com base nos slides da Aula 3, que destacam a **limpeza como a fundação invisível** de qualquer dashboard confiável. O objetivo aqui não é apenas "fazer funcionar", mas tomar decisões conscientes sobre tipos, valores ausentes, duplicidades, variáveis derivadas e exportação da base limpa. fileciteturn2file0

## Regras desta atividade
- Você deve **construir os códigos**.
- O notebook orienta os passos, mas não entrega as soluções prontas.
- Sempre que fizer uma decisão de limpeza, **documente o porquê** em comentário ou célula markdown.
- Ao final, exporte uma base limpa para uso nas próximas aulas.

## Dataset desta atividade
Arquivo bruto: `vendas_brasil_aula3_bruto.csv`


## 1. Preparação do ambiente

Importe as bibliotecas necessárias para trabalhar com:
- leitura de dados
- tratamento de nulos
- identificação de infinitos
- exportação do resultado final

**Sugestão:** Pandas e NumPy já resolvem toda a atividade.


In [2]:
import pandas as pd

import numpy as np

## 2. Leitura da base bruta

Leia o arquivo `vendas_brasil_aula3_bruto.csv` em um DataFrame chamado `df`.

Depois:
1. Exiba as primeiras linhas
2. Exiba as últimas linhas
3. Observe visualmente se já existem sinais de sujeira


In [3]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

print("Primeiras linhas:")
print(df.head())

print("\nÚltimas linhas:")
print(df.tail())

Primeiras linhas:
   pedido_id        data  uf        canal    categoria           produto  \
0       1000  2024/04/29  SP  Loja Física  Informática      Notebook Pro   
1       1001  2024-06-17  PR          NaN  Informática      Notebook Pro   
2       1002  2024-05-27  PR  Marketplace  Periféricos  Teclado Mecânico   
3       1003  2024-07-08  SP  Marketplace  Informática      Notebook Pro   
4       1004  2024-05-20  RS       Online  Informática      Notebook Pro   

  cliente_id  quantidade   receita    lucro      observacao  
0       C134           1   4535.11  1289.83              ok  
1       C106           1   4005.59   541.36  entrega rápida  
2       C131           1    309.02   128.26  entrega rápida  
3       C148           2   7943.78  1574.09              ok  
4       C105           5  19926.65  4281.84              ok  

Últimas linhas:
     pedido_id        data  uf        canal    categoria       produto  \
223       1180  2024-03-11  SC  Loja Física    Telefonia  Smar

## 3. Check-up inicial do dataset

Com base no checklist de um dataset "clean" apresentado na aula, faça um diagnóstico inicial da base. fileciteturn2file0

### Sua missão
Use comandos que permitam responder:
- Qual é o tamanho da base?
- Quais são os tipos atuais das colunas?
- Existem valores nulos?
- Há colunas com tipo inadequado?
- Há algo que pareça impedir análises confiáveis?


In [4]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

print("Shape (linhas, colunas):")
print(df.shape)

print("\nTipos de dados:")
print(df.dtypes)

print("\nInfo geral:")
df.info()

print("\nValores nulos por coluna:")
print(df.isna().sum())

print("\nValores infinitos:")
print(np.isinf(df.select_dtypes(include=[float, int])).sum())

Shape (linhas, colunas):
(228, 11)

Tipos de dados:
pedido_id       int64
data           object
uf             object
canal          object
categoria      object
produto        object
cliente_id     object
quantidade      int64
receita        object
lucro         float64
observacao     object
dtype: object

Info geral:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228 entries, 0 to 227
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   pedido_id   228 non-null    int64  
 1   data        228 non-null    object 
 2   uf          223 non-null    object 
 3   canal       223 non-null    object 
 4   categoria   223 non-null    object 
 5   produto     228 non-null    object 
 6   cliente_id  228 non-null    object 
 7   quantidade  228 non-null    int64  
 8   receita     226 non-null    object 
 9   lucro       222 non-null    float64
 10  observacao  178 non-null    object 
dtypes: float64(1), int64(2), object(8)
m

## 4. Batalha 1 — A tirania dos tipos de dados

Nos slides, vimos que datas não podem ser tratadas como texto e que números em formato string quebram análises. fileciteturn2file0

### Tarefa
Converta corretamente, quando fizer sentido:
- `data`
- `receita`
- `lucro`
- `quantidade`

### Orientação
- Investigue valores estranhos antes de converter
- Pense no uso de `errors='coerce'`
- Registre em markdown ou comentário quais problemas encontrou


In [5]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

df['data'] = pd.to_datetime(df['data'], errors='coerce')

def limpar_moeda(col):
    return (
        col.astype(str)
        .str.replace('R\$', '', regex=True)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
    )

df['receita'] = pd.to_numeric(limpar_moeda(df['receita']), errors='coerce')
df['lucro'] = pd.to_numeric(limpar_moeda(df['lucro']), errors='coerce')

df['quantidade'] = pd.to_numeric(df['quantidade'], errors='coerce')

print(df.dtypes)
print(df.head())

pedido_id              int64
data          datetime64[ns]
uf                    object
canal                 object
categoria             object
produto               object
cliente_id            object
quantidade             int64
receita              float64
lucro                float64
observacao            object
dtype: object
   pedido_id       data  uf        canal    categoria           produto  \
0       1000 2024-04-29  SP  Loja Física  Informática      Notebook Pro   
1       1001        NaT  PR          NaN  Informática      Notebook Pro   
2       1002        NaT  PR  Marketplace  Periféricos  Teclado Mecânico   
3       1003        NaT  SP  Marketplace  Informática      Notebook Pro   
4       1004        NaT  RS       Online  Informática      Notebook Pro   

  cliente_id  quantidade    receita     lucro      observacao  
0       C134           1   453511.0  128983.0              ok  
1       C106           1   400559.0   54136.0  entrega rápida  
2       C131           1

<>:8: SyntaxWarning: invalid escape sequence '\$'
<>:8: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_223/2724252491.py:8: SyntaxWarning: invalid escape sequence '\$'
  .str.replace('R\$', '', regex=True)


### Reflexão curta
Explique:
1. Por que deixar `data` como texto pode quebrar análises temporais?
2. Por que deixar `receita` como string pode distorcer cálculos?


Deixar a data como texto impede operações temporais, como ordenação correta, filtragem por período e cálculos de tendência, pois o sistema não reconhece valores como datas reais.

Já manter a receita como string impede operações matemáticas, como soma e média, além de poder gerar resultados incorretos ou erros, já que o Python trata esses valores como texto e não como números.

## 5. Batalha 2 — O enigma dos valores ausentes

A aula reforça que valores ausentes não devem ser tratados automaticamente; a decisão precisa ser justificada. fileciteturn2file0

### Tarefa
1. Mapeie os valores ausentes por coluna
2. Identifique quais colunas críticas têm nulos
3. Defina uma estratégia para cada caso:
   - remover linhas?
   - preencher?
   - manter?
4. Justifique cada escolha

### Dica
Evite decisões automáticas sem análise de contexto.


In [6]:
nulos = df.isna().sum()

print("Valores ausentes por coluna:")
print(nulos)

percentual_nulos = (df.isna().sum() / len(df)) * 100
print("\nPercentual de nulos (%):")
print(percentual_nulos)

Valores ausentes por coluna:
pedido_id       0
data          224
uf              5
canal           5
categoria       5
produto         0
cliente_id      0
quantidade      0
receita         5
lucro           6
observacao     50
dtype: int64

Percentual de nulos (%):
pedido_id      0.000000
data          98.245614
uf             2.192982
canal          2.192982
categoria      2.192982
produto        0.000000
cliente_id     0.000000
quantidade     0.000000
receita        2.192982
lucro          2.631579
observacao    21.929825
dtype: float64


### Registro reflexivo
Escreva aqui sua justificativa para o tratamento dos nulos:
- Em quais colunas você removeu linhas?
- Em quais colunas você preencheu valores?
- Em quais situações decidiu manter o nulo?


Remoção de linhas:
Linhas com valores nulos na coluna data foram removidas, pois sem essa informação não é possível realizar análises temporais, tornando o registro pouco útil.
Preenchimento de valores:
Nas colunas categóricas, como canal e estado, os valores nulos foram preenchidos com “Desconhecido”, permitindo manter os registros sem comprometer agrupamentos e análises.
Manutenção de valores nulos:
Em colunas numéricas críticas, como receita, lucro e quantidade, os valores nulos foram mantidos (ou avaliados para possível remoção posterior), evitando distorções nos cálculos caso fossem preenchidos artificialmente.

## 6. Batalha 3 — O ataque dos clones (duplicidades)

A aula alerta que nem toda duplicidade é erro automático: pode haver compra legítima repetida ou dupla inserção do sistema. fileciteturn2file0

### Tarefa
1. Investigue se há linhas duplicadas
2. Analise se a duplicidade parece nociva para os KPIs
3. Escolha uma estratégia:
   - remover duplicidades totais?
   - remover apenas com base em certas colunas?
   - manter?
4. Justifique a decisão


In [8]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

# 1. Identificar duplicatas

duplicadas_total = df.duplicated().sum()
print("Duplicatas totais:", duplicadas_total)

print("\nExemplos de duplicatas completas:")
print(df[df.duplicated()].head())

cols_chave = ['data', 'produto', 'quantidade', 'receita']
duplicadas_parciais = df.duplicated(subset=cols_chave).sum()

print("\nDuplicatas por colunas-chave:", duplicadas_parciais)

# 2. Análise no próprio código
# Se duplicatas completas existem, elas provavelmente são erro de sistema
# (mesmo registro inserido mais de uma vez)

# 3. Estratégia adotada

# Remover duplicatas completas
df = df.drop_duplicates()

# NÃO remover duplicatas parciais automaticamente,
# pois podem representar vendas reais repetidas

# =========================
# 4. Verificação após limpeza
# =========================
print("\nDuplicatas após remoção:", df.duplicated().sum())

# 5. Comentário (justificativa)
# Removemos duplicatas completas pois indicam erro de registro e podem inflar KPIs.
# Mantivemos duplicatas parciais pois podem representar transações legítimas.

Duplicatas totais: 7

Exemplos de duplicatas completas:
     pedido_id        data   uf        canal    categoria       produto  \
220       1132  2024-01-29   PR  Marketplace    Telefonia  Smartphone X   
221       1148  2024-03-18   mg  Loja Física  Informática  Notebook Pro   
222       1093  2024-05-13   BA  Marketplace  Informática    Monitor 27   
223       1180  2024-03-11   SC  Loja Física    Telefonia  Smartphone X   
225       1115  2024-07-29   MG  Loja Física  Informática  Notebook Pro   

    cliente_id  quantidade   receita    lucro       observacao  
220       C123           3   8870.79  2326.37               ok  
221       C179           5   21240.2  6245.67   entrega rápida  
222       C170           2   2699.72   724.07  cliente premium  
223       C158           1   2563.44   415.42              NaN  
225       C144           3  12566.04  2879.76              NaN  

Duplicatas por colunas-chave: 7

Duplicatas após remoção: 0


## 7. Padronização de categorias

Os slides mostram que categorias duplicadas ou mal escritas podem distorcer rankings e filtros. fileciteturn2file0

### Tarefa
Inspecione e padronize, se necessário:
- `uf`
- `canal`
- `categoria`

### Pense em:
- maiúsculas e minúsculas
- espaços extras
- acentuação / variações
- categorias equivalentes escritas de formas diferentes


In [9]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

# =========================
# 1. INSPEÇÃO INICIAL
# =========================

print("UF (antes):")
print(df['uf'].value_counts(dropna=False))

print("\nCanal (antes):")
print(df['canal'].value_counts(dropna=False))

print("\nCategoria (antes):")
print(df['categoria'].value_counts(dropna=False))


# =========================
# 2. PADRONIZAÇÃO
# =========================

# --- UF ---
df['uf'] = (
    df['uf']
    .astype(str)
    .str.strip()
    .str.upper()
)

# --- CANAL ---
df['canal'] = (
    df['canal']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'online': 'Online',
        'e-commerce': 'Online',
        'ecommerce': 'Online',
        'marketplace': 'Marketplace',
        'market place': 'Marketplace',
        'loja fisica': 'Loja Física',
        'loja física': 'Loja Física'
    })
)

# --- CATEGORIA ---
df['categoria'] = (
    df['categoria']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'eletronico': 'Eletrônicos',
        'eletrônicos': 'Eletrônicos',
        'eletronicos': 'Eletrônicos',
        'informatica': 'Informática',
        'informática': 'Informática'
    })
)

# =========================
# 3. INSPEÇÃO FINAL
# =========================

print("\nUF (depois):")
print(df['uf'].value_counts())

print("\nCanal (depois):")
print(df['canal'].value_counts())

print("\nCategoria (depois):")
print(df['categoria'].value_counts())


# =========================
# 4. COMENTÁRIO (JUSTIFICATIVA)
# =========================
# Padronizamos textos removendo espaços, ajustando maiúsculas/minúsculas
# e consolidando categorias equivalentes. Isso evita fragmentação dos dados,
# garantindo análises e rankings mais confiáveis.

UF (antes):
uf
ES     39
SC     37
MG     32
PR     26
RS     25
RJ     24
BA     19
SP     16
NaN     5
 mg     3
rj      1
BA      1
Name: count, dtype: int64

Canal (antes):
canal
Loja Física    74
Online         71
Marketplace    70
NaN             5
ONLINE          4
Loja fisica     3
MarketPlace     1
Name: count, dtype: int64

Categoria (antes):
categoria
Telefonia      61
Informática    60
Periféricos    51
Áudio          48
NaN             5
telefonia       2
Perifericos     1
Name: count, dtype: int64

UF (depois):
uf
ES     39
SC     37
MG     35
PR     26
RJ     25
RS     25
BA     20
SP     16
NAN     5
Name: count, dtype: int64

Canal (depois):
canal
Loja Física    77
Online         75
Marketplace    71
nan             5
Name: count, dtype: int64

Categoria (depois):
categoria
telefonia      63
Informática    60
periféricos    51
áudio          48
nan             5
perifericos     1
Name: count, dtype: int64


## 8. Subindo de nível — Criação de variáveis derivadas

Depois da limpeza, é hora de enriquecer a base com variáveis úteis para análise. fileciteturn2file0

### Tarefa
Crie, se possível:
- `ano`
- `mes`
- `ano_mes`
- `margem_lucro`

### Atenção
A criação de `margem_lucro` exige cuidado com divisões por zero e possíveis valores infinitos.


In [10]:
df = pd.read_csv('vendas_brasil_aula3_bruto.csv')

# Garantir que data e valores já estejam tratados (se necessário)
df['data'] = pd.to_datetime(df['data'], errors='coerce')

# =========================
# 1. Variáveis de tempo
# =========================

df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month
df['ano_mes'] = df['data'].dt.to_period('M')

# =========================
# 2. Margem de lucro
# =========================
# Fórmula: lucro / receita

df['margem_lucro'] = df['lucro'] / df['receita']

# Tratar divisões problemáticas (infinitos e NaN)
df['margem_lucro'] = df['margem_lucro'].replace([np.inf, -np.inf], np.nan)

# =========================
# Ver resultado
# =========================
print(df[['data', 'ano', 'mes', 'ano_mes', 'receita', 'lucro', 'margem_lucro']].head())

TypeError: unsupported operand type(s) for /: 'float' and 'str'

### Reflexão técnica
Explique:
1. O que pode acontecer ao calcular margem de lucro quando a receita é zero?
2. Como você decidiu tratar esse caso?


Quando a receita é zero, o cálculo da margem de lucro (lucro / receita) gera uma divisão por zero, resultando em valores infinitos (inf) ou indefinidos (NaN). Isso pode distorcer análises, médias e visualizações, além de gerar erros em algumas operações.

Para tratar esse caso, substituí os valores infinitos por NaN, utilizando .replace([np.inf, -np.inf], np.nan). Essa abordagem evita distorções nos indicadores e permite lidar com esses casos de forma controlada, como ignorá-los em cálculos ou analisá-los separadamente.

## 9. Seleção final — Menos é mais

A aula propõe que nem toda coluna precisa seguir para a base analítica final. fileciteturn2file0

### Tarefa
Crie um DataFrame final, por exemplo `df_clean` ou `df_dash`, contendo apenas as colunas estritamente necessárias para análises de negócio.

**Sugestão de raciocínio:**
- Quais colunas ajudam a responder perguntas de negócio?
- Quais colunas são operacionais, auxiliares ou dispensáveis?
- O que precisa existir para as próximas aulas de visualização e dashboard?


In [11]:
# Seleção das colunas relevantes para análise
cols_final = [
    'data',
    'ano',
    'mes',
    'ano_mes',
    'uf',
    'canal',
    'produto',
    'categoria',
    'quantidade',
    'receita',
    'lucro',
    'margem_lucro'
]

# Criar DataFrame final
df_clean = df[cols_final].copy()

# Visualizar
print(df_clean.head())
print("\nShape final:", df_clean.shape)

KeyError: "['margem_lucro'] not in index"

## 10. Validação final da base limpa

Antes de exportar, faça uma checagem final.

### Verifique:
- os tipos estão corretos?
- ainda há nulos problemáticos?
- ainda há duplicidades nocivas?
- existe algum infinito em `margem_lucro`?
- a base está pronta para ser usada em análises e dashboards?


In [13]:
# =========================
# 1. Tipos de dados
# =========================
print("Tipos de dados:")
print(df_clean.dtypes)

# =========================
# 2. Valores nulos
# =========================
print("\nValores nulos por coluna:")
print(df_clean.isna().sum())

# =========================
# 3. Duplicidades
# =========================
duplicadas = df_clean.duplicated().sum()
print("\nDuplicatas:", duplicadas)

# =========================
# 4. Valores infinitos (margem_lucro)
# =========================
inf_margem = np.isinf(df_clean['margem_lucro']).sum()
print("\nInfinitos em margem_lucro:", inf_margem)

# =========================
# 5. Verificação geral
# =========================
print("\nResumo:")
if duplicadas == 0 and inf_margem == 0:
    print("Base pronta para análise ✅")
else:
    print("Atenção: ainda existem problemas ⚠️")

Tipos de dados:


NameError: name 'df_clean' is not defined

## 11. Exportação da base "Clean"

Agora gere o arquivo final da base tratada.

### Tarefa
Exporte sua base limpa com o nome:

`vendas_brasil_aula3_clean.csv`

### Observação
Esse arquivo será o selo de qualidade do seu trabalho desta aula.


In [16]:
# Exportar base limpa
df_clean.to_csv('vendas_brasil_aula3_bruto.csv', index=False)

print("Arquivo exportado com sucesso!")

NameError: name 'df_clean' is not defined

## 12. Conclusão e registro reflexivo

Escreva um pequeno texto respondendo:

1. Quais foram os principais problemas de qualidade encontrados?
2. Qual decisão de limpeza foi mais difícil?
3. Que impacto essas falhas poderiam causar em um dashboard?
4. Por que a etapa de limpeza é considerada a "fundação invisível" do projeto?


Durante a análise, foram identificados diversos problemas de qualidade, como valores nulos, categorias inconsistentes, dados numéricos armazenados como texto e possíveis duplicidades. Esses problemas exigiram diferentes estratégias de tratamento para garantir a integridade dos dados.

A decisão mais difícil foi lidar com valores ausentes em colunas críticas, como receita e lucro, pois qualquer preenchimento inadequado poderia distorcer os resultados. Por isso, foi necessário avaliar cuidadosamente quando manter, remover ou tratar esses dados.

Essas falhas poderiam impactar diretamente um dashboard, gerando indicadores incorretos, análises distorcidas e decisões equivocadas. Pequenos erros nos dados podem resultar em grandes interpretações erradas.

A etapa de limpeza é considerada a “fundação invisível” porque, embora não seja visível no resultado final, é essencial para garantir que todas as análises e visualizações sejam confiáveis e precisas. Sem uma base limpa, qualquer insight perde credibilidade.

## 13. Desafio extra (opcional)

Use sua base limpa para responder rapidamente:

- Qual UF concentra maior receita?
- Qual canal gera maior lucro?
- Existe algum mês com desempenho claramente acima dos demais?

Você pode responder com tabelas simples ou pequenos agrupamentos.


In [ ]:
# Desafio extra opcional
